In [1]:
import os
from MDToolkit.IO.read_file import cif_file_to_frame
from MDToolkit.IO.write_file import  write_lammps_data_file
from MDToolkit.custom_systems.membranes import  create_2D_membrane, create_pore
from MDToolkit.custom_systems.liquids import create_water_box
from MDToolkit.paths import CIF_FILES, OUTPUT

In [2]:
#####################################################
# Nanoporous Membrane Reservoir System
#####################################################

membrane_unit_cell_file_name = "MoS2.cif"

membrane_dims = [50, 50]

pore_radius = 5.0

output_dir = OUTPUT
output_file_name = "output.data"

# Creating Nanoporous Membrane

membrane_unit = os.path.join(CIF_FILES, membrane_unit_cell_file_name)
membrane = cif_file_to_frame(membrane_unit)
membrane = create_2D_membrane(membrane, membrane_dims, [0, 90, 0])
membrane.set_center([0.0, 0.0, 0.0])
membrane = create_pore(membrane, pore_radius, [0.0, 0.0], maintain_stoichiometry = True)

# Creating Reservoirs
water_box_bounds_1 = {
  "min_x" : -104, "max_x" : -54,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_bounds_2 = {
  "min_x" : 54, "max_x" : 104,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_1 = create_water_box(water_box_bounds_1, "SPCE", density_correction=0.56)
water_box_2 = create_water_box(water_box_bounds_2, "SPCE", density_correction=0.56)


# Writing Output File
write_lammps_data_file(membrane, output_file_name, output_dir, atom_style = "atomic")

[np.float64(8.667299975), np.float64(26.001899925000004)]
Pore radius: 5.00 Å
H2O Template File: /workspace/code/MDToolkit/MDToolkit/data/pdb_files/H2O_SPCE.pdb


UnicodeDecodeError: 'ascii' codec can't decode byte 0xef in position 0: ordinal not in range(128)

In [ ]:
#####################################################
# CNT Graphene Water System
#####################################################

# Import CNT
CNT_file_path = os.path.join(PDB_FILES, "CNT_16_16_10.pdb")
CNT_system = pdb_file_to_structured_system(CNT_file_path)
CNT_system.rotate_system_spherical(90, 0, 0)
CNT_system.combine_all_molecules()
CNT_system.set_COM_to_origin()

# Generate Graphene Layers 
graphene_cif_file_path = os.path.join(CIF_FILES, "graphene.cif")
membrane_dims = [52, 52, 0]
monolayer_graphene_system = cif_file_to_monolayer_membrane(graphene_cif_file_path, max_dimension = membrane_dims, shrink_buffer=[2.0, 0.71, 0.71])
monolayer_graphene_system.set_COM_to_origin()
monolayer_graphene_system, pore_area = create_pore_circular(monolayer_graphene_system, radius = 10.65)
monolayer_graphene_system.combine_all_molecules()

monolayer_graphene_system_copy = deepcopy(monolayer_graphene_system)

# Combine CNT and Graphenes
monolayer_graphene_system.set_COM_to_origin([-52, 0, 0])
CNT_system.combine_with_other_structured_system(monolayer_graphene_system)
monolayer_graphene_system_copy.set_COM_to_origin([52, 0, 0])
CNT_system.combine_with_other_structured_system(monolayer_graphene_system_copy)
CNT_system.combine_all_molecules()

# Create Water Boxes
water_box_bounds_1 = {
  "min_x" : -104, "max_x" : -54,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_bounds_2 = {
  "min_x" : 54, "max_x" : 104,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_1 = create_water_box(water_box_bounds_1, "SPCE", density_correction=0.56)
water_box_2 = create_water_box(water_box_bounds_2, "SPCE", density_correction=0.56)
CNT_system.combine_with_other_structured_system(water_box_1)
CNT_system.combine_with_other_structured_system(water_box_2)
CNT_system.populate_elemental_properties_for_all_atoms()

for i in range(1, len(CNT_system.molecule_list)):
  CNT_system.molecule_list[i].populate_bonds_from_molecular_data(read_molecular_data_json_entry(alias_key = "H2O"))
  CNT_system.molecule_list[i].populate_angles_from_molecular_data(read_molecular_data_json_entry(alias_key = "H2O"))

# Write Structure

write_lammps_structure_file_atomic_full(CNT_system, file_name = "CNT1616_Graphene_Water.data")

In [2]:
#####################################################
# Graphene Water Salt System
#####################################################

# Generate Graphene Layers 
graphene_cif_file_path = os.path.join(CIF_FILES, "graphene.cif")
membrane_dims = [51, 51, 0]
monolayer_graphene_system = cif_file_to_monolayer_membrane(graphene_cif_file_path, max_dimension = membrane_dims, shrink_buffer=[2.0, 0.71, 0.71])
monolayer_graphene_system.set_COM_to_origin()
monolayer_graphene_system, pore_area = create_pore_circular(monolayer_graphene_system, radius = 5.0)
monolayer_graphene_system.combine_all_molecules()

# Create Water Boxes
water_box_bounds_1 = {
  "min_x" : -52.0, "max_x" : -2.0,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}
water_box_bounds_2 = {
  "min_x" :  2.0, "max_x" : 52.0,
  "min_y" : -25.0, "max_y" : 25.0,
  "min_z" : -25.0, "max_z" : 25.0
}

aq_salt1 = create_simplesalt_in_water_box(water_box_bounds_1, "KCl", 1.0, solution_density=1.04, H2O_geometry="SPCE", density_correction=0.2)
aq_salt2 = create_simplesalt_in_water_box(water_box_bounds_2, "KCl", 1.0, solution_density=1.04, H2O_geometry="SPCE", density_correction=0.2)

monolayer_graphene_system.combine_with_other_structured_system(aq_salt1)
monolayer_graphene_system.combine_with_other_structured_system(aq_salt2)

monolayer_graphene_system.populate_elemental_properties_for_all_atoms()

write_lammps_structure_file_atomic_full(monolayer_graphene_system, file_name="Graphene_Water_KCl.data")

H2O Template File: /workspace/code/MDToolkit/MDToolkit/data/pdb_files/H2O_SPCE.pdb
Packmol input file successfully written to /workspace/code/MDToolkit/packmol_input_files/H2O_Salt_box_packmol_helper.inp

################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                             Version 21.2.3 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Seed for random number generator:        24184
  Output file: /workspace/code/MDToolkit/Output/Salt_H2O_box.pdb
  Reading coordinate file: /workspace/code/MDToolkit/MDToolkit/data/pdb_files/H2O_SPCE.pdb
  Reading coordinate file: /w